In [1]:
import pytest
import ipytest
from builder import *

In [2]:
ipytest.autoconfig()

In [3]:
%%ipytest

@pytest.fixture
def base_design():
    """Returns a valid design dictionary for testing."""
    return {
        "nodes": {
            "sensors": {
                "positions": np.array([[0,0,0], [1,1,1], [2,2,2], [3,3,3]]),
                "uplink_type": [0, 1, 0, 1], # 0=Optical, 1=RF (2 of each)
                "nT": [[0,0,1], [0,0,1], [0,0,1], [0,0,1]],
                "nR": [[0,0,1], [0,0,1], [0,0,1], [0,0,1]],
                "m": [1, 1, 1, 1],
                "rx_area": 1e-4,
                "tx_power": 0.5,
                "battery_capacity_J": 1000,
                "initial_charge": 1.0
            },
            "masters": {
                "positions": np.array([[2.5, 2.0, 3.0]]),
                "rx_area": 1e-3,
                "sensitivity": -95
            }
        },
        "TIA": {"gain": 1e4, "noise": 1e-12}
    }

# --- 1. Parsing and Defaults ---

def test_node_builder_defaults():
    """Verify that defaults (FOV, BW) are applied when missing from dictionary."""
    simple_design = {
        "nodes": {
            "masters": {
                "positions": np.array([0, 0, 0]),
            }
        }
    }
    nb = NodeBuilder(simple_design, "masters")
    assert nb.FOV == np.pi/2
    assert nb.BW == 10000
    assert nb.IR_pass_filter is True

def test_node_builder_type_parsing(base_design):
    """Ensure masters and sensors extract their specific parameters correctly."""
    # Test Masters
    nb_m = NodeBuilder(base_design, "masters")
    assert nb_m.sensitivity == -95
    assert nb_m.IR_pass_filter is True
    
    # Test Sensors
    nb_s = NodeBuilder(base_design, "sensors")
    assert nb_s.VLC_pass_filter is True
    assert nb_s.battery_capacity_J == 1000

# --- 2. Sanity Check and Hybrid Logic ---

def test_hybrid_uplink_separation(base_design):
    """
    Critical Test: Ensure that nT and m are sliced to match ONLY 
    the nodes with optical uplinks (uplink_type == 0).
    """
    nb = NodeBuilder(base_design, "sensors")
    
    # Total positions were 4
    assert nb.positions.shape[0] == 4
    
    # But only 2 had uplink_type=0 (Optical)
    assert nb.no_optical_uplinks == 2
    assert nb.no_RF_uplinks == 2
    
    # nT and m should have been sliced to size 2
    assert nb.nT.shape == (2, 3)
    assert nb.m.shape == (2,)

def test_scalar_uplink_handling(base_design):
    """Verify that a scalar uplink_type is correctly expanded to match N positions."""
    # Modify design to have 4 positions but a single scalar for uplink
    base_design["nodes"]["sensors"]["uplink_type"] = 0 
    
    nb = NodeBuilder(base_design, "sensors")
    assert len(nb.uplink_type) == 4
    assert np.all(nb.uplink_type == 0)
    assert nb.no_optical_uplinks == 4

# --- 3. Geometric / Shape Integrity ---

def test_position_reshaping():
    """Ensure positions always become (N, 3) even if input is a flat list."""
    design = {
        "nodes": {
            "ambient_nodes": {
                "positions": np.array([1, 2, 3]) # Flat 1D input
            }
        }
    }
    nb = NodeBuilder(design, "ambient_nodes")
    assert nb.positions.shape == (1, 3)

.....                                                                                        [100%]
5 passed in 0.04s
